# EXP04: SL x Trail 交叉 + K-Fold + 参数悬崖

**合并**: NB08 + NB09

1. SL [3.5,4.0,4.5,5.0] x TrailAct [0.6,0.8,1.0] x TrailDist [0.20,0.25,0.30] = 36 变体
2. 自动选冠军做 K-Fold
3. +/-1步参数悬崖检测

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import pandas as pd

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: SL x Trail 网格

In [ ]:
results = []
for sl in [3.5, 4.0, 4.5, 5.0]:
    for t_act in [0.6, 0.8, 1.0]:
        for t_dist in [0.20, 0.25, 0.30]:
            kw = {**LIVE_KWARGS, "sl_atr_mult": sl,
                  "trailing_activate_atr": t_act, "trailing_distance_atr": t_dist}
            r = run_variant(data, f"SL{sl}_T{t_act}/D{t_dist}", **kw)
            r['sl'] = sl
            r['trail_act'] = t_act
            r['trail_dist'] = t_dist
            results.append(r)

print_ranked(results)

In [ ]:
df = pd.DataFrame([{'sl': r['sl'], 'trail_act': r['trail_act'], 'trail_dist': r['trail_dist'],
                     'sharpe': r['sharpe'], 'total_pnl': r['total_pnl'],
                     'max_dd': r.get('max_dd', 0)} for r in results])
print("\nBy SL (avg across trail combos):")
print(df.groupby('sl')[['sharpe', 'total_pnl', 'max_dd']].mean().round(2))

## Part 2: K-Fold 验证

In [ ]:
best = sorted(results, key=lambda r: r['sharpe'], reverse=True)[0]
BEST_SL = best['sl']
BEST_T_ACT = best['trail_act']
BEST_T_DIST = best['trail_dist']
print(f"Champion: SL={BEST_SL}, TrailAct={BEST_T_ACT}, TrailDist={BEST_T_DIST}, Sharpe={best['sharpe']:.2f}")

champion_kwargs = {
    **LIVE_KWARGS,
    "sl_atr_mult": BEST_SL,
    "trailing_activate_atr": BEST_T_ACT,
    "trailing_distance_atr": BEST_T_DIST,
}

print("\n=== Champion K-Fold ===")
champ_folds = run_kfold(data, champion_kwargs, label_prefix="Champ_")
print("\n=== Baseline K-Fold ===")
base_folds = run_kfold(data, LIVE_KWARGS, label_prefix="Base_")

for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  Champ={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

## Part 3: 参数悬崖检测

In [ ]:
cliff_results = []
for dt_act in [-0.1, 0, 0.1]:
    for dt_dist in [-0.05, 0, 0.05]:
        t_act = round(BEST_T_ACT + dt_act, 2)
        t_dist = round(BEST_T_DIST + dt_dist, 2)
        if t_act <= 0 or t_dist <= 0:
            continue
        kw = {**LIVE_KWARGS, "trailing_activate_atr": t_act, "trailing_distance_atr": t_dist, "sl_atr_mult": BEST_SL}
        r = run_variant(data, f"T{t_act}/D{t_dist}", **kw)
        cliff_results.append(r)

print_ranked(cliff_results)
sharpes = [r['sharpe'] for r in cliff_results]
cliff = max(sharpes) - min(sharpes)
print(f"\nParameter cliff (Sharpe range): {cliff:.2f} {'CLIFF!' if cliff > 0.5 else 'smooth'}")

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp04_results.json', 'w') as f:
    json.dump(sanitize_for_json(results), f, indent=2)
print('Saved')